# 第03章：内存操作深入 — 指针、Load、Store

## 前置知识
- 第01-02章

## 学习目标
- 深入理解 Triton 的指针运算模型
- 掌握 `tl.load` 和 `tl.store` 的所有参数
- 学会处理非连续内存（strided access）
- 理解边界处理的各种场景

In [ ]:
import torch
import triton
import triton.language as tl

## 3.1 Triton 指针模型

在 CUDA 中，指针就是一个内存地址。在 Triton 中也一样，但操作的粒度不同：

```
CUDA:   单个指针 → 单个元素
         float* ptr;  ptr[i] = value;

Triton: 指针向量 → 一块元素
         ptr + offsets → 一组地址
         tl.load(ptr + offsets) → 一块数据
```

### 指针运算示意

```
内存:    [v0] [v1] [v2] [v3] [v4] [v5] [v6] [v7]
地址:     0    1    2    3    4    5    6    7

ptr = 数组起始地址
offsets = tl.arange(0, 4) = [0, 1, 2, 3]

ptr + offsets = [ptr+0, ptr+1, ptr+2, ptr+3]
                  ↓       ↓       ↓       ↓
tl.load(...)  = [v0,     v1,     v2,     v3]  ← 加载一块数据！
```

## 3.2 tl.load 完整参数

```python
tl.load(
    pointer,          # 指针或指针向量
    mask=None,        # 布尔 mask，False 的位置不加载
    other=None,       # mask 为 False 时的默认值
    cache_modifier="", # 缓存提示: ".ca", ".cg", ".cv"
    eviction_policy="", # 驱逐策略: "evict_first", "evict_last"
    volatile=False,    # 是否标记为 volatile
)
```

### 缓存修饰符 (Cache Modifier)

| 修饰符 | 含义 | 使用场景 |
|--------|------|----------|
| `""` (默认) | 默认缓存策略 | 大多数情况 |
| `".ca"` | Cache at all levels | 数据会被重复使用 |
| `".cg"` | Cache at global level | 数据在 L2 缓存 |
| `".cv"` | Don't cache (volatile) | 流式数据，只用一次 |

### 驱逐策略

| 策略 | 含义 |
|------|------|
| `"evict_first"` | 优先驱逐（不太重要的数据） |
| `"evict_last"` | 最后驱逐（重要数据保留更久） |

In [ ]:
# 示例：不同的 mask 和 other 行为
@triton.jit
def load_demo_kernel(in_ptr, out_ptr, n, BLOCK_SIZE: tl.constexpr):
    """
    演示 mask 和 other 参数:
    - 对于有效位置：加载真实数据
    - 对于越界位置：填充 -1.0
    """
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    
    # mask: 哪些位置是有效的
    mask = offsets < n
    
    # other=-1.0: 越界位置填充 -1.0（而不是读取垃圾数据）
    data = tl.load(in_ptr + offsets, mask=mask, other=-1.0)
    
    # 注意：store 也需要 mask，否则会写入越界内存！
    tl.store(out_ptr + offsets, data, mask=mask)

# 演示
n = 5  # 只有5个有效元素
BLOCK_SIZE = 8  # 但 block 大小是 8
x = torch.arange(5, dtype=torch.float32, device='cuda')
out = torch.full((BLOCK_SIZE,), -999.0, dtype=torch.float32, device='cuda')

load_demo_kernel[(1,)](x, out, n, BLOCK_SIZE=BLOCK_SIZE)
print(f"输入 (n=5): {x.tolist()}")
print(f"输出 (BLOCK=8): {out.tolist()}")
print(f"→ 前5个是真实数据，后3个保持 -999（因为 store 也有 mask）")

## 3.3 Strided 内存访问

不是所有数据都是连续存储的。比如访问矩阵的一列：

```
矩阵 A (3×4) Row-Major:
内存: [a00, a01, a02, a03, a10, a11, a12, a13, a20, a21, a22, a23]
       ↑                   ↑                   ↑
       第0列               第0列               第0列
       offset=0            offset=4            offset=8

stride_m = 4 (行步长)
要访问第0列: offsets = [0, 4, 8] = arange(0,3) * stride_m
```

In [ ]:
@triton.jit
def column_sum_kernel(
    a_ptr, out_ptr,
    M, N, stride_m, stride_n,
    BLOCK_M: tl.constexpr,
):
    """
    计算矩阵每列的和: out[j] = sum(A[:, j])
    
    每个 program 处理一列，需要按 stride_m 步长访问非连续内存。
    """
    col_idx = tl.program_id(0)  # 当前处理第几列
    
    # 访问第 col_idx 列的所有行
    row_offsets = tl.arange(0, BLOCK_M)
    mask = row_offsets < M
    
    # 关键: 非连续访问！
    # A[i, col_idx] 的地址 = a_ptr + i * stride_m + col_idx * stride_n
    ptrs = a_ptr + row_offsets * stride_m + col_idx * stride_n
    
    col_data = tl.load(ptrs, mask=mask, other=0.0)
    col_sum = tl.sum(col_data, axis=0)
    
    tl.store(out_ptr + col_idx, col_sum)

# 测试列求和
M, N = 100, 8
a = torch.randn(M, N, device='cuda')
out = torch.empty(N, device='cuda')

# 取大于等于 M 的最小2的幂
BLOCK_M = triton.next_power_of_2(M)

column_sum_kernel[(N,)](
    a, out, M, N,
    a.stride(0), a.stride(1),
    BLOCK_M=BLOCK_M,
)

expected = a.sum(dim=0)
print(f"列求和正确性: {torch.allclose(out, expected, atol=1e-4)}")
print(f"结果: {out[:4].tolist()}")
print(f"参考: {expected[:4].tolist()}")

## 3.4 2D Block 加载模式

加载一个 2D 子矩阵块是 GEMM 和 Attention 的核心操作：

```
大矩阵 A (M×K):              加载的子块 (BLOCK_M × BLOCK_K):
┌───────────────────┐         ┌─────────┐
│                   │         │ a00 a01  │
│     ┌─────────┐   │    →    │ a10 a11  │
│     │  子块    │   │         │ a20 a21  │
│     └─────────┘   │         └─────────┘
│                   │
└───────────────────┘

地址计算:
block_ptr = a_ptr + row_start * stride_m + col_start * stride_n
offsets = row_offsets[:, None] * stride_m + col_offsets[None, :] * stride_n
data = tl.load(block_ptr + offsets, mask=mask)
```

In [ ]:
@triton.jit
def load_block_2d_kernel(
    a_ptr, out_ptr,
    M, N, stride_am, stride_an,
    start_row, start_col,  # 子块起始位置
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    """
    从大矩阵中加载一个 BLOCK_M x BLOCK_N 的子块。
    这是 GEMM 和 Attention 的基本操作。
    """
    row_offsets = tl.arange(0, BLOCK_M)
    col_offsets = tl.arange(0, BLOCK_N)
    
    # 计算全局行列索引
    rows = start_row + row_offsets
    cols = start_col + col_offsets
    
    # 2D mask
    mask = (rows[:, None] < M) & (cols[None, :] < N)
    
    # 2D 偏移: 广播 (BLOCK_M, 1) + (1, BLOCK_N) → (BLOCK_M, BLOCK_N)
    offsets = rows[:, None] * stride_am + cols[None, :] * stride_an
    
    block = tl.load(a_ptr + offsets, mask=mask, other=0.0)
    
    # 存入连续的输出矩阵
    out_offsets = row_offsets[:, None] * BLOCK_N + col_offsets[None, :]
    out_mask = (row_offsets[:, None] < BLOCK_M) & (col_offsets[None, :] < BLOCK_N)
    tl.store(out_ptr + out_offsets, block, mask=out_mask)

# 测试：从大矩阵中提取子块
M, N = 10, 10
BLOCK_M, BLOCK_N = 4, 4
a = torch.arange(M * N, dtype=torch.float32, device='cuda').reshape(M, N)
out = torch.zeros(BLOCK_M, BLOCK_N, device='cuda')

# 提取从 (2, 3) 开始的 4x4 子块
start_row, start_col = 2, 3
load_block_2d_kernel[(1,)](
    a, out, M, N, a.stride(0), a.stride(1),
    start_row, start_col,
    BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N,
)

print("原矩阵:")
print(a.int())
print(f"\n提取子块 [{start_row}:{start_row+BLOCK_M}, {start_col}:{start_col+BLOCK_N}]:")
print(out.int())
print(f"\nPyTorch 参考:")
print(a[start_row:start_row+BLOCK_M, start_col:start_col+BLOCK_N].int())

## 3.5 `tl.make_block_ptr` — 更优雅的 2D 访问

手动计算 2D 偏移虽然灵活，但容易出错。Triton 提供了 `tl.make_block_ptr` 来简化：

```python
block_ptr = tl.make_block_ptr(
    base=a_ptr,           # 基地址
    shape=(M, N),         # 完整矩阵的形状
    strides=(stride_m, stride_n),  # 步长
    offsets=(row_start, col_start), # 子块起始偏移
    block_shape=(BLOCK_M, BLOCK_N), # 子块大小
    order=(1, 0),         # 内存序 (1,0)=row-major, (0,1)=col-major
)
data = tl.load(block_ptr, boundary_check=(0, 1))  # 自动边界检查
```

`make_block_ptr` 的好处：
1. 不需要手动计算 2D 偏移和 mask
2. `boundary_check` 自动处理越界
3. 可以用 `tl.advance(block_ptr, (dm, dn))` 移动指针
4. 编译器可以更好地优化内存访问

In [ ]:
@triton.jit
def block_ptr_demo_kernel(
    a_ptr, out_ptr,
    M, N,
    stride_am, stride_an,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    """
    使用 make_block_ptr 加载 2D 子块。
    比手动计算偏移更简洁、更安全。
    """
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    
    # 创建 block pointer
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr,
        shape=(M, N),
        strides=(stride_am, stride_an),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N),
        order=(1, 0),  # row-major
    )
    
    # 加载 —— boundary_check 自动处理越界，填充 0
    block = tl.load(a_block_ptr, boundary_check=(0, 1))
    
    # 输出也用 block_ptr
    out_block_ptr = tl.make_block_ptr(
        base=out_ptr,
        shape=(M, N),
        strides=(stride_am, stride_an),
        offsets=(pid_m * BLOCK_M, pid_n * BLOCK_N),
        block_shape=(BLOCK_M, BLOCK_N),
        order=(1, 0),
    )
    tl.store(out_block_ptr, block * 2.0, boundary_check=(0, 1))  # 乘以2

# 测试
M, N = 7, 5  # 非对齐尺寸
BLOCK_M, BLOCK_N = 4, 4
a = torch.arange(M * N, dtype=torch.float32, device='cuda').reshape(M, N)
out = torch.zeros_like(a)

grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
block_ptr_demo_kernel[grid](
    a, out, M, N, a.stride(0), a.stride(1),
    BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N,
)

print(f"输入:\n{a}")
print(f"\n输出 (×2):\n{out}")
print(f"\n正确性: {torch.allclose(out, a * 2)}")

## 3.6 `tl.advance` — 移动 Block Pointer

在 GEMM 的 K 维循环中，我们需要沿 K 方向移动指针：

```
矩阵 A (M×K):     K方向 →
┌─────┬─────┬─────┐
│ blk0│ blk1│ blk2│  ← 每次 advance(0, BLOCK_K) 移到下一块
│     │     │     │
└─────┴─────┴─────┘

a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))
                                       ↑      ↑
                                    行不变  列移动 BLOCK_K
```

In [ ]:
@triton.jit
def advance_demo_kernel(
    a_ptr, out_ptr,
    M, K,
    stride_am, stride_ak,
    BLOCK_M: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    """
    演示 tl.advance: 沿 K 维遍历并累加。
    计算每行的元素之和: out[i] = sum(A[i, :])
    """
    pid = tl.program_id(0)  # 行块索引
    
    a_block_ptr = tl.make_block_ptr(
        base=a_ptr,
        shape=(M, K),
        strides=(stride_am, stride_ak),
        offsets=(pid * BLOCK_M, 0),  # 从第 0 列开始
        block_shape=(BLOCK_M, BLOCK_K),
        order=(1, 0),
    )
    
    # 累加器
    acc = tl.zeros((BLOCK_M,), dtype=tl.float32)
    
    # 沿 K 维循环
    for k in range(0, K, BLOCK_K):
        block = tl.load(a_block_ptr, boundary_check=(0, 1))
        acc += tl.sum(block, axis=1)  # 沿 K 维求和
        a_block_ptr = tl.advance(a_block_ptr, (0, BLOCK_K))  # 移动到下一个 K 块
    
    # 存储结果
    row_offsets = pid * BLOCK_M + tl.arange(0, BLOCK_M)
    mask = row_offsets < M
    tl.store(out_ptr + row_offsets, acc, mask=mask)

# 测试
M, K = 10, 32
BLOCK_M, BLOCK_K = 4, 8
a = torch.randn(M, K, device='cuda')
out = torch.empty(M, device='cuda')

grid = (triton.cdiv(M, BLOCK_M),)
advance_demo_kernel[grid](
    a, out, M, K, a.stride(0), a.stride(1),
    BLOCK_M=BLOCK_M, BLOCK_K=BLOCK_K,
)

expected = a.sum(dim=1)
print(f"行求和正确性: {torch.allclose(out, expected, atol=1e-4)}")

## 总结

| 操作 | API | 说明 |
|------|-----|------|
| 指针偏移 | `ptr + offsets` | 基本的指针运算 |
| 加载 | `tl.load(ptr, mask, other)` | 从 GPU 内存读取 |
| 存储 | `tl.store(ptr, val, mask)` | 写入 GPU 内存 |
| 块指针 | `tl.make_block_ptr(...)` | 结构化 2D 访问 |
| 移动指针 | `tl.advance(ptr, offsets)` | 移动 block pointer |
| 缓存控制 | `cache_modifier` | 控制 L1/L2 缓存行为 |

### 选择指南
- **简单 1D 访问**: `ptr + tl.arange(...)` + `mask`
- **2D 子块访问**: `tl.make_block_ptr` + `boundary_check`
- **需要循环移动**: `tl.make_block_ptr` + `tl.advance`

## 练习

1. **gather 操作**: 实现 `out[i] = data[indices[i]]`（间接索引加载）
2. **scatter 操作**: 实现 `data[indices[i]] = values[i]`
3. **带步长的加载**: 从矩阵中每隔一行加载数据（stride=2*stride_m）
4. **block_ptr 矩阵复制**: 用 make_block_ptr 实现矩阵的完整复制